# Phase 6 · Architecture 2: LegalSeq2Seq

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','-q','install','bert-score','rouge-score'])
from pathlib import Path
import os, json, math, random, sys, pickle, subprocess, zipfile, time
from collections import Counter, defaultdict
import cloudpickle, torch
from torch import nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download

ARCH = 2
BASE = Path('/kaggle/input/datasets/abdakt/a2-p3')
WEIGHTS, BPE = BASE / 'best.pt', BASE / 'tokenizer.json'
OUT = Path('/kaggle/working/phase6_a2_final_1536')
OUT.mkdir(exist_ok=True)
METHODS = ['lora', 'lorma', 'pfeiffer', 'houlsby']
SOURCE, TARGET, BATCH = 1536, 640, 1
EPOCHS, PATIENCE, EVAL_STEPS = 8, 7, 250
SEED = 781
tok = Tokenizer.from_file(str(BPE))
PAD = tok.token_to_id('<pad>')
BOS = tok.token_to_id('<bos>')
EOS = tok.token_to_id('<eos>')
VOCAB = tok.get_vocab_size()

In [2]:
def read_rows(split):
    path = hf_hub_download('Exploration-Lab/CS-781-Capstone',
        f'Phase_6_Summ/{split.title()}/{split}.json', repo_type='dataset',
        token=os.environ['HF_TOKEN'])
    rows = json.loads(Path(path).read_text())
    if isinstance(rows, dict):
        rows = [dict(v, id=v.get('id', k)) if isinstance(v, dict) else {'id': k, 'document': v}
                for k, v in rows.items()]
    return rows

def prepare(split):
    data = []
    for row in read_rows(split):
        doc = tok.encode(row['document'], add_special_tokens=False).ids
        budget = SOURCE - 1
        if len(doc) > budget:
            quarter = budget // 4
            starts = [0, (len(doc)-quarter)//3, 2*(len(doc)-quarter)//3]
            doc = sum((doc[i:i+quarter] for i in starts), []) + doc[-(budget-3*quarter):]
        target = []
        if split == 'train':
            target = tok.encode(row['summary'], add_special_tokens=False).ids
            if len(target) < TARGET:
                target.append(EOS)
            else:
                target = target[:TARGET]
        data.append((row['id'], doc + [BOS], target))
    return data

rows = prepare('train')
ids = sorted({str(r[0]) for r in rows})
random.Random(SEED).shuffle(ids)
held = set(ids[:max(1, round(.1*len(ids)))])
train = [r for r in rows if str(r[0]) not in held]
dev = [r for r in rows if str(r[0]) in held]
test = prepare('test')
dev_sample = random.Random(SEED).sample(dev, min(32,len(dev)))
wanted = {str(r[0]) for r in dev_sample}
refs = {str(r['id']):r['summary'] for r in read_rows('train') if str(r['id']) in wanted}
with (OUT/'calibration.pkl').open('wb') as f:
    pickle.dump((dev_sample,refs),f)
with (OUT/'data.pkl').open('wb') as f:
    pickle.dump((train,dev,test), f)
print(f'{len(train):,} train | {len(dev):,} dev | {len(test):,} test')

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

6,327 train | 703 dev | 100 test


In [3]:
# Architecture 2: LegalSeq2Seq — RoPE, RMSNorm, SwiGLU, cross-attention
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


class RMSNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        z = x.float()
        z = z * torch.rsqrt(z.square().mean(-1, keepdim=True) + 1e-5)
        return z.to(x.dtype) * self.weight


class SelfAttention(nn.Module):
    def __init__(self, cfg, causal):
        super().__init__()
        self.heads = cfg['heads']
        self.head_dim = cfg['dim'] // self.heads
        self.causal = causal
        self.qkv = nn.Linear(cfg['dim'], 3 * cfg['dim'], bias=False)
        self.out = nn.Linear(cfg['dim'], cfg['dim'], bias=False)
        inv = 10000 ** (-torch.arange(0, self.head_dim, 2).float() / self.head_dim)
        self.register_buffer('inv_freq', inv, persistent=False)

    def rope(self, x, positions):
        phase = positions.float().unsqueeze(-1) * self.inv_freq.float()
        cos, sin = phase.cos()[None, None].to(x.dtype), phase.sin()[None, None].to(x.dtype)
        even, odd = x[..., ::2], x[..., 1::2]
        return torch.stack((even * cos - odd * sin, even * sin + odd * cos), -1).flatten(-2)

    def forward(self, x, mask=None, past=None, use_cache=False):
        b, t, d = x.shape
        q, k, v = self.qkv(x).view(b, t, 3, self.heads, self.head_dim).unbind(2)
        q, k, v = (z.transpose(1, 2) for z in (q, k, v))
        old = 0 if past is None else past[0].size(2)
        positions = torch.arange(old, old + t, device=x.device)
        q, k = self.rope(q, positions), self.rope(k, positions)
        if past is not None:
            k, v = torch.cat((past[0], k), 2), torch.cat((past[1], v), 2)
        allowed = None if mask is None else mask[:, None, None, :]
        fast_causal = self.causal and past is None and mask is None
        if self.causal and not fast_causal:
            causal_mask = torch.arange(k.size(2), device=x.device)[None, :] <= positions[:, None]
            allowed = causal_mask[None, None] if allowed is None else allowed & causal_mask[None, None]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed,
                                          is_causal=fast_causal, dropout_p=0.0)
        y = self.out(y.transpose(1, 2).contiguous().view(b, t, d))
        return y, (k, v) if use_cache else None


class CrossAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.heads, self.head_dim = cfg['heads'], cfg['dim'] // cfg['heads']
        self.q = nn.Linear(cfg['dim'], cfg['dim'], bias=False)
        self.kv = nn.Linear(cfg['dim'], 2 * cfg['dim'], bias=False)
        self.out = nn.Linear(cfg['dim'], cfg['dim'], bias=False)

    def project_memory(self, memory):
        b, s, _ = memory.shape
        k, v = self.kv(memory).view(b, s, 2, self.heads, self.head_dim).unbind(2)
        return k.transpose(1, 2), v.transpose(1, 2)

    def forward(self, x, memory, mask, cached=None):
        b, t, d = x.shape
        q = self.q(x).view(b, t, self.heads, self.head_dim).transpose(1, 2)
        k, v = self.project_memory(memory) if cached is None else cached
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask[:, None, None, :], dropout_p=0.0)
        return self.out(y.transpose(1, 2).contiguous().view(b, t, d))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, f = cfg['dim'], cfg['ffn_dim']
        self.gate = nn.Linear(d, f, bias=False)
        self.up = nn.Linear(d, f, bias=False)
        self.down = nn.Linear(f, d, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class Bottleneck(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.down = nn.Linear(width, 32)
        self.up = nn.Linear(32, width)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        x = x + self.up(F.relu(self.down(x)))
        if hasattr(self, "ff_adapter"): x = self.ff_adapter(x)
        return x

class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(cfg['dim']), RMSNorm(cfg['dim'])
        self.attn = SelfAttention(cfg, causal=False)
        self.ffn = FeedForward(cfg)

    def forward(self, x, mask):
        x = x + self.attn(self.norm1(x), mask)[0]
        if hasattr(self, "att_adapter"): x = self.att_adapter(x)
        x = x + self.ffn(self.norm2(x))
        if hasattr(self, "ff_adapter"): x = self.ff_adapter(x)
        return x


class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.norm2, self.norm3 = (RMSNorm(cfg['dim']) for _ in range(3))
        self.attn = SelfAttention(cfg, causal=True)
        self.cross = CrossAttention(cfg)
        self.ffn = FeedForward(cfg)

    def forward(self, x, memory, source_mask, target_mask=None, past=None, cross_kv=None, use_cache=False):
        a, cache = self.attn(self.norm1(x), target_mask, past, use_cache)
        x = x + a
        if hasattr(self, "att_adapter"): x = self.att_adapter(x)
        x = x + self.cross(self.norm2(x), memory, source_mask, cross_kv)
        x = x + self.ffn(self.norm3(x))
        if hasattr(self, "ff_adapter"): x = self.ff_adapter(x)
        return x, cache


class LegalSeq2Seq(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = dict(cfg)
        self.embed = nn.Embedding(cfg['vocab_size'], cfg['dim'])
        self.encoder = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg['encoder_layers'])])
        self.decoder = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg['decoder_layers'])])
        self.encoder_norm, self.decoder_norm = RMSNorm(cfg['dim']), RMSNorm(cfg['dim'])
        self.apply(self._init)
        for block in self.encoder:
            for weight in (block.attn.out.weight, block.ffn.down.weight):
                nn.init.normal_(weight, std=0.02 / math.sqrt(2 * len(self.encoder)))
        for block in self.decoder:
            for weight in (block.attn.out.weight, block.cross.out.weight, block.ffn.down.weight):
                nn.init.normal_(weight, std=0.02 / math.sqrt(3 * len(self.decoder)))

    @staticmethod
    def _init(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)

    def encode(self, source, source_mask):
        x = self.embed(source)
        for block in self.encoder:
            if self.training and self.cfg['gradient_checkpointing']:
                x = checkpoint(block, x, source_mask, use_reentrant=False)
            else:
                x = block(x, source_mask)
        return self.encoder_norm(x)

    def decode(self, target, memory, source_mask, target_mask=None, past=None, cross_kv=None, use_cache=False):
        x, caches = self.embed(target), []
        for i, block in enumerate(self.decoder):
            if self.training and self.cfg['gradient_checkpointing']:
                def layer(z, mem, module=block):
                    return module(z, mem, source_mask, target_mask)[0]
                x = checkpoint(layer, x, memory, use_reentrant=False)
            else:
                x, cache = block(x, memory, source_mask, target_mask,
                                 None if past is None else past[i],
                                 None if cross_kv is None else cross_kv[i], use_cache)
                caches.append(cache)
        return self.decoder_norm(x), caches

    def forward(self, source, target, labels=None):
        source_mask = source.ne(self.cfg['pad_id'])
        memory = self.encode(source, source_mask)
        x, _ = self.decode(target, memory, source_mask, target.ne(self.cfg['pad_id']))
        if labels is None:
            return F.linear(x, self.embed.weight)
        h, y = x.flatten(0, 1), labels.flatten()
        total = h.new_zeros((), dtype=torch.float32)
        for start in range(0, len(y), 256):
            def part(z, truth):
                return F.cross_entropy(F.linear(z, self.embed.weight).float(), truth,
                                       ignore_index=-100, reduction='sum')
            if self.training:
                total = total + checkpoint(part, h[start:start+256], y[start:start+256], use_reentrant=False)
            else:
                total = total + part(h[start:start+256], y[start:start+256])
        return total

In [4]:
class Adapt(nn.Module):
    def __init__(self, base, method):
        super().__init__()
        self.base, self.method = base, method
        if method == 'scale':
            self.scale = nn.Parameter(torch.ones(base.out_features))
        else:
            width = base.out_features if method == 'lorma' else base.in_features
            self.A = nn.Linear(width, 8, bias=False)
            self.B = nn.Linear(8, base.out_features, bias=False)
            nn.init.zeros_(self.B.weight)

    def forward(self, x):
        y = self.base(x)
        if self.method == 'scale':
            return y * self.scale.to(y.dtype)
        return y + self.B(self.A(y if self.method == 'lorma' else x))

In [5]:
def build(method):
    ck = torch.load(WEIGHTS, map_location='cpu', weights_only=False)
    cfg = dict(ck['config'], gradient_checkpointing=True, pad_id=PAD)
    m = LegalSeq2Seq(cfg)
    m.load_state_dict(ck['model'])
    m.requires_grad_(False)
    blocks = list(m.encoder) + list(m.decoder)
    width = cfg.get('dim',cfg.get('width'))
    if method in ('houlsby','pfeiffer'):
        for b in blocks:
            b.ff_adapter = Bottleneck(width)
            if method == 'houlsby': b.att_adapter = Bottleneck(width)
    else:
        for parent in list(m.modules()):
            for name, layer in list(parent.named_children()):
                if isinstance(layer, nn.Linear) and name in ('qkv','q','kv','out'):
                    setattr(parent, name, Adapt(layer, method))
    return m

In [6]:
def logits(m, batch):
    docs = pad([r[1] for r in batch], PAD)
    y = pad([r[2] for r in batch], -100)
    prev = pad([[BOS]+r[2][:-1] for r in batch], PAD)
    return m(docs,prev)[y != -100], y[y != -100]

In [7]:
def pad(rows, value):
    return nn.utils.rnn.pad_sequence([torch.tensor(r) for r in rows], batch_first=True,
                                     padding_value=value).cuda()

def batches(rows, seed=None):
    order = list(range(len(rows)))
    if seed is not None: random.Random(seed).shuffle(order)
    for start in range(0, len(order), 64):
        chunk = sorted(order[start:start+64], key=lambda i: len(rows[i][1])+len(rows[i][2]))
        for j in range(0,len(chunk),BATCH):
            yield [rows[i] for i in chunk[j:j+BATCH]]

def load_best(method):
    m = build(method)
    m.load_state_dict(torch.load(OUT/method/'best.pt',map_location='cpu',weights_only=True),strict=False)
    return m.cuda().eval()

def train_one(method):
    torch.manual_seed(SEED)
    with (OUT/'data.pkl').open('rb') as f: train, dev, _ = pickle.load(f)
    dest = OUT/method
    dest.mkdir(exist_ok=True)
    m = build(method).cuda()
    params = {k:p for k,p in m.named_parameters() if p.requires_grad}
    teacher = load_best('lora') if method == 'distill' else None
    lr = 2e-5 if method in ('partial','decoder','cross','distill') else 3e-4
    opt = torch.optim.AdamW(params.values(),lr=lr,fused=True)
    scaler = torch.amp.GradScaler('cuda')
    best, stale, step = float('inf'), 0, 0
    start = time.monotonic()
    print(method, 'trainable', sum(p.numel() for p in params.values()), flush=True)
    for epoch in range(EPOCHS):
        for j, batch in enumerate(batches(train, SEED+epoch), 1):
            m.train()
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.float16):
                z,y = logits(m,batch)
                loss = F.cross_entropy(z.float(),y)
                if teacher is not None:
                    with torch.no_grad(): t,_ = logits(teacher,batch)
                    kd = F.kl_div((z.float()/2).log_softmax(-1),(t.float()/2).softmax(-1),reduction='batchmean')
                    loss = .5*loss + 2*kd
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params.values(),1.)
            scaler.step(opt)
            scaler.update()
            step += 1
            if step % EVAL_STEPS and j != math.ceil(len(train)/BATCH): continue
            m.eval()
            total, count = torch.zeros((), device='cuda'), 0
            with torch.inference_mode(), torch.autocast('cuda',dtype=torch.float16):
                for v in batches(dev):
                    z,y = logits(m,v)
                    total += F.cross_entropy(z.float(),y,reduction='sum')
                    count += len(y)
            score = total.item()/count
            print(method, 'epoch',epoch+1,'step',step,'dev NLL',round(score,4),flush=True)
            if score < best:
                best, stale = score, 0
                torch.save({k:p.detach().half().cpu() for k,p in params.items()},dest/'best.tmp')
                os.replace(dest/'best.tmp',dest/'best.pt')
            else: stale += 1
            if stale >= PATIENCE: break
        if stale >= PATIENCE: break
    (dest/'done').write_text(json.dumps(dict(dev_nll=best,steps=step,seconds=time.monotonic()-start,
                    peak_GiB=torch.cuda.max_memory_allocated()/2**30)))

In [8]:
@torch.inference_mode()
@torch.autocast('cuda',dtype=torch.float16)
def generate(m,doc,limit=TARGET):
    ids = torch.tensor([doc],device='cuda')
    mask = ids.ne(PAD)
    memory = m.encode(ids,mask)
    kv = [b.cross.project_memory(memory) for b in m.decoder]
    def decode(token,cache):
        h,cache = m.decode(token,memory,mask,past=cache,cross_kv=kv,use_cache=True)
        return F.linear(h[:,-1],m.embed.weight),cache
    z,cache = decode(ids.new_full((1,1),BOS),None)
    result, followers = [], defaultdict(set)
    for i in range(limit):
        z = z[:, :VOCAB].float()
        for special in (PAD,BOS):
            if special != EOS: z[:,special] = -torch.inf
        if i < 8: z[:,EOS] = -torch.inf
        if len(result)>=2:
            z[:,list(followers[tuple(result[-2:])])] = -torch.inf
        token = z.argmax(-1)
        n = token.item()
        if n == EOS: break
        if len(result)>=2: followers[tuple(result[-2:])].add(n)
        result.append(n)
        z,cache = decode(token[:,None],cache)
    return result

def predict(method):
    m = load_best(method)
    tokenizer = Tokenizer.from_file(str(BPE))
    with (OUT/'data.pkl').open('rb') as f: _,_,test = pickle.load(f)
    from rouge_score.rouge_scorer import RougeScorer
    from bert_score import score as bertscore
    scorer = RougeScorer(['rouge1','rougeL'],use_stemmer=True)
    with (OUT/'calibration.pkl').open('rb') as f: sample,refs = pickle.load(f)
    generated = [generate(m,d) for _,d,_ in sample]
    gold = [refs[str(k)] for k,_,_ in sample]
    caps = [192,384,TARGET]
    texts = [[tokenizer.decode(t[:cap]).strip() for t in generated] for cap in caps]
    _,_,f1 = bertscore(sum(texts,[]),gold*len(caps),lang='en',model_type='roberta-large',
                       batch_size=4,device='cuda',verbose=False)
    metrics = []
    for cap,group,bert in zip(caps,texts,f1.reshape(len(caps),-1).mean(1).tolist()):
        rs = [scorer.score(ref,hyp) for ref,hyp in zip(gold,group)]
        metrics.append(dict(cap=cap,bert=bert,**{k:sum(r[k].fmeasure for r in rs)/len(rs)
                                               for k in ('rouge1','rougeL')}))
    peak = max(r['bert'] for r in metrics)
    chosen = max((r for r in metrics if r['bert'] >= peak-.002),
                 key=lambda r:r['rouge1']+r['rougeL'])
    (OUT/method/'decoding.json').write_text(json.dumps(dict(selected=chosen,results=metrics),indent=2))
    print(method,chosen,flush=True)
    rows = [dict(id=k,summary=tokenizer.decode(generate(m,d,chosen["cap"])).strip()) for k,d,_ in test]
    package(method,rows)

def package(method,rows):
    with zipfile.ZipFile(OUT/f'submission_{method}.zip','w',zipfile.ZIP_DEFLATED) as z:
        z.writestr('prediction.json',json.dumps(rows,ensure_ascii=False))

In [9]:
def run_jobs(names, worker):
    jobs = {}
    queue = list(names)
    payload = OUT/'worker.pkl'
    payload.write_bytes(cloudpickle.dumps(worker))
    command = "import cloudpickle,sys,torch;torch.set_num_threads(1);cloudpickle.load(open(sys.argv[1],'rb'))(sys.argv[2])"
    while queue or jobs:
        for gpu in (0,1):
            if gpu in jobs:
                name,p = jobs[gpu]
                if p.poll() is None: continue
                if p.returncode: raise RuntimeError(f'{name} exited with {p.returncode}')
                del jobs[gpu]
            ready = [n for n in queue if worker != train_one or n != 'distill' or (OUT/'lora'/'done').exists()]
            if ready:
                name = ready[0]
                queue.remove(name)
                print(f'GPU {gpu} | {name}',flush=True)
                jobs[gpu] = name,subprocess.Popen([sys.executable,'-c',command,str(payload),name],
                    env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu)))
        if queue and not jobs: raise RuntimeError('Train lora before distill.')
        if jobs: time.sleep(2)
    for _, p in jobs.values():
        p.wait()


In [10]:
run_jobs(METHODS, train_one)

GPU 0 | lora
GPU 1 | lorma
lorma trainable 1376256
lora trainable 1093632
lorma epoch 1 step 250 dev NLL 3.1031
lora epoch 1 step 250 dev NLL 3.1262
lorma epoch 1 step 500 dev NLL 3.0736
lora epoch 1 step 500 dev NLL 3.0847
lorma epoch 1 step 750 dev NLL 3.057
lora epoch 1 step 750 dev NLL 3.0604
lorma epoch 1 step 1000 dev NLL 3.0467
lora epoch 1 step 1000 dev NLL 3.0453
lorma epoch 1 step 1250 dev NLL 3.0467
lora epoch 1 step 1250 dev NLL 3.0389
lorma epoch 1 step 1500 dev NLL 3.0371
lora epoch 1 step 1500 dev NLL 3.0297
lorma epoch 1 step 1750 dev NLL 3.0345
lora epoch 1 step 1750 dev NLL 3.024
lorma epoch 1 step 2000 dev NLL 3.0354
lora epoch 1 step 2000 dev NLL 3.0208
lorma epoch 1 step 2250 dev NLL 3.027
lora epoch 1 step 2250 dev NLL 3.0145
lorma epoch 1 step 2500 dev NLL 3.0208
lora epoch 1 step 2500 dev NLL 3.0089
lorma epoch 1 step 2750 dev NLL 3.0202
lora epoch 1 step 2750 dev NLL 3.0058
lorma epoch 1 step 3000 dev NLL 3.0128
lora epoch 1 step 3000 dev NLL 3.0014
lorma epoch

In [11]:
run_jobs(METHODS, predict)

GPU 0 | lora
GPU 1 | lorma


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 2535.11it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


lorma {'cap': 640, 'bert': 0.8267855644226074, 'rouge1': 0.3760968446865356, 'rougeL': 0.2026521573919483}


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 2524.45it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


lora {'cap': 640, 'bert': 0.8278728723526001, 'rouge1': 0.41656457076051495, 'rougeL': 0.21338819787788252}
GPU 1 | pfeiffer
GPU 0 | houlsby


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 2726.84it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


pfeiffer {'cap': 384, 'bert': 0.8270312547683716, 'rouge1': 0.3884280878414635, 'rougeL': 0.2042740904352022}


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 2566.77it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


houlsby {'cap': 640, 'bert': 0.8250757455825806, 'rouge1': 0.4125762745296486, 'rougeL': 0.21025983730284287}


In [12]:
def overlap(a,b):
    a,b = Counter(a.lower().split()),Counter(b.lower().split())
    return 2*sum((a&b).values())/max(1,sum(a.values())+sum(b.values()))

candidates = defaultdict(list)
for name in METHODS:
    with zipfile.ZipFile(OUT/f'submission_{name}.zip') as z:
        for row in json.loads(z.read('prediction.json')):
            candidates[row['id']].append(row['summary'])
ensemble = []
for key,_,_ in test:
    group = candidates[key]
    scores = [sum(overlap(s,t) for j,t in enumerate(group) if j != i) for i,s in enumerate(group)]
    ensemble.append(dict(id=key, summary=group[max(range(len(group)),key=scores.__getitem__)]))
package('ensemble',ensemble)
print(OUT)

/kaggle/working/phase6_a2_final_1536


## Notes